In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
# pip install pyarrow and dask
import dask.dataframe as dd
plt.rcParams['figure.dpi'] = 300
sns.set(rc={'axes.facecolor':'grey'})

In [2]:
# Importing the data
region = ['south_sudan', 'eastern_east_africa', 'eastern_ukraine', 'southern_africa', 'west_africa', 'sri_lanka', 'lake_victoria_basin']
model = ['CanESM5', 'CCSM4', 'CESM1', 'GEM5', 'GFDL', 'NASA', 'NCEP', 'CMCC', 'DWD', 'ECMWF', 'METEO', 'JMA']

# Create an empty dictionary to store the DataFrames
dataframes = {}

for i in region:
    for j in model:
    # Construct the filename
        filename = f'data/netCDF/{i}_{j}_merged.nc'  # Using f-string for clarity
        try:
            # Load the NetCDF file, convert to DataFrame, and store in the dictionary
            dataframes[(i, j)] = xr.open_mfdataset(filename).to_dataframe().reset_index()

        except FileNotFoundError:
            print(f"Warning: File not found: {filename}")
        except Exception as e:  # Catch other potential errors
            print(f"Error loading file {filename}: {e}")

In [3]:
# Adding Dry Masking
dry_mask_months = {'south_sudan': [1,2,12],
                   'eastern_east_africa': [1,2],
                   'southern_africa': [5,6,7,8,9],
                   'west_africa': [1,2,3,11,12]}

In [4]:
#heatmap
def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    d = data.pivot(index=args[1], columns=args[0], values=args[2])
    sns.heatmap(d, **kwargs, vmin=-0.7, vmax=0.7,
                cmap=sns.color_palette('vlag', 15),
                linewidths=0.1, linecolor='black')
    plt.xticks(np.arange(0.5, 12.5, 1))  # Set xticks explicitly
    plt.gca().set_xticklabels(np.arange(0.5, 12.5, 1))  # Set xticklabels
    plt.xticks(fontsize=6)
    plt.yticks(fontsize=6)
    plt.gca().invert_yaxis()

In [5]:
# Create empty dictionaries to store results
ensemble_means_corr_dict = {}
start_year = 1993
end_year = 2024

region = ['south_sudan', 'eastern_east_africa', 'eastern_ukraine', 'southern_africa', 'west_africa', 'sri_lanka', 'lake_victoria_basin']
model = ['CanESM5', 'CCSM4', 'CESM1', 'GEM5', 'GFDL', 'NASA', 'NCEP', 'CMCC', 'DWD', 'ECMWF', 'METEO', 'JMA']

for i in region:
    for j in model:
        # Construct the key for the dataframes dictionary
        df_key = (i, j)

        # Check if the DataFrame exists for this region/model combination
        if df_key in dataframes:
            df = dataframes[df_key].dropna()
            mask = (df['time'].dt.year >= start_year) & (df['time'].dt.year <= end_year)
            df_filtered = df[mask]

            # Calculate ensemble mean
            ensemble_means = (df_filtered
                              .groupby(['time', 'lead_time', 'latitude', 'longitude'])[['predicted_precip', 'precip']]
                              .mean().reset_index())

            # Extract months
            ensemble_means['month'] = ensemble_means['time'].dt.month

            # Calculate the correlation
            ensemble_means_corr = (ensemble_means
                                   .groupby(['month', 'lead_time'])[['predicted_precip', 'precip']].corr(method='spearman')
                                   .iloc[0::2, -1].droplevel(-1).reset_index())  # Corrected droplevel

            # Store results in dictionaries
            ensemble_means_corr_dict[df_key] = ensemble_means_corr

        else:
            print(f"Warning: DataFrame not found for region '{i}' and model '{j}'")

In [6]:
# Combine all correlations into one DataFrame
ensemble_corr = pd.concat(ensemble_means_corr_dict.values(), keys=ensemble_means_corr_dict.keys(), names=['region', 'model'])
ensemble_corr = ensemble_corr.rename(columns = {'precip': 'corr'}).reset_index().drop('level_2', axis=1)

In [7]:
# Makes copy of corr table
ensemble_corr_dm = ensemble_corr.copy()

for index, row in ensemble_corr.iterrows():  # Iterate over rows
    region = row['region']
    month = row['month']

    if region in dry_mask_months and month in dry_mask_months[region]:
        ensemble_corr_dm.loc[index, 'corr'] = np.nan  # Set 'corr' to NaN for the matching row

In [8]:
fg = sns.FacetGrid(ensemble_corr, col='region', row='model', sharex=False, sharey=False)
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'corr', square = True)

fg.set_titles('Spearman Corr of Ensemble Mean \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")
fg.savefig('figures/ensemble_means_corr.png')
plt.close()

In [9]:
fg = sns.FacetGrid(ensemble_corr_dm, col='region', row='model', sharex=False, sharey=False)
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'corr', square = True)
fg.set_titles('Spearman Corr of Ensemble Mean \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")
fg.savefig('figures/dry_masked_ensemble_means_corr.png')
plt.close()

In [10]:
# Create empty dictionaries to store results
spatial_means_corr_dict = {}
start_year = 1993
end_year = 2024

region = ['south_sudan', 'eastern_east_africa', 'eastern_ukraine', 'southern_africa', 'west_africa', 'sri_lanka', 'lake_victoria_basin']
model = ['CanESM5', 'CCSM4', 'CESM1', 'GEM5', 'GFDL', 'NASA', 'NCEP', 'CMCC', 'DWD', 'ECMWF', 'METEO', 'JMA']

for i in region:
    for j in model:
        df_key = (i, j)

        if df_key in dataframes:
            df = dataframes[df_key].dropna()
            mask = (df['time'].dt.year >= start_year) & (df['time'].dt.year <= end_year)
            df_filtered = df[mask]

            # Calculate ensemble mean (same as before, but now using the df variable)
            ensemble_means = (df_filtered
                              .groupby(['time', 'lead_time', 'latitude', 'longitude'])[['predicted_precip', 'precip']]
                              .mean().reset_index())

            # Calculate spatial means
            spatial_means = (ensemble_means  # Use the ensemble_means DataFrame
                              .groupby(['time', 'lead_time'])[['predicted_precip', 'precip']]
                              .mean().reset_index())

            # Extract months
            spatial_means['month'] = spatial_means['time'].dt.month

            # Calculate correlation
            spatial_means_corr = (spatial_means
                                   .groupby(['month', 'lead_time'])[['predicted_precip', 'precip']].corr(method='spearman')
                                   .iloc[0::2, -1].droplevel(-1).reset_index()) #Corrected droplevel

            # Store results in dictionaries
            spatial_means_corr_dict[df_key] = spatial_means_corr

        else:
            print(f"Warning: DataFrame not found for region '{i}' and model '{j}'")

In [11]:
# Combine all correlations into one DataFrame
spatial_corr = pd.concat(spatial_means_corr_dict.values(), keys=spatial_means_corr_dict.keys(), names=['region', 'model'])
spatial_corr = spatial_corr.rename(columns = {'precip': 'corr'}).reset_index().drop('level_2', axis=1)

In [12]:
# Makes copy of corr table
spatial_corr_dm = spatial_corr.copy()

for index, row in spatial_corr.iterrows():  # Iterate over rows
    region = row['region']
    month = row['month']

    if region in dry_mask_months and month in dry_mask_months[region]:
        spatial_corr_dm.loc[index, 'corr'] = np.nan  # Set 'corr' to NaN for the matching row

In [13]:
fg = sns.FacetGrid(spatial_corr, col='region', row='model', sharex=False, sharey=False)
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'corr', square = True)

fg.set_titles('Spearman Corr of Spatial Mean \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")
fg.savefig('figures/spatial_means_corr.png')
plt.close()

In [14]:
fg = sns.FacetGrid(spatial_corr_dm, col='region', row='model', sharex=False, sharey=False)
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'corr', square = True)
fg.set_titles('Spearman Corr of Spatial Mean \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")
fg.savefig('figures/dry_masked_spatial_means_corr.png')
plt.close()

In [15]:
# Create empty dictionaries to store results
temporal_means_corr_dict = {}
start_year = 1993
end_year = 2024

region = ['south_sudan', 'eastern_east_africa', 'eastern_ukraine', 'southern_africa', 'west_africa', 'sri_lanka', 'lake_victoria_basin']
model = ['CanESM5', 'CCSM4', 'CESM1', 'GEM5', 'GFDL', 'NASA', 'NCEP', 'CMCC', 'DWD', 'ECMWF', 'METEO', 'JMA']

for i in region:
    for j in model:
        df_key = (i, j)

        if df_key in dataframes:
            df = dataframes[df_key].dropna()
            mask = (df['time'].dt.year >= start_year) & (df['time'].dt.year <= end_year)
            df_filtered = df[mask]

            # Calculate ensemble mean (reuse from previous examples)
            ensemble_means = (df_filtered
                              .groupby(['time', 'lead_time', 'latitude', 'longitude'])[['predicted_precip', 'precip']]
                              .mean().reset_index())
            ensemble_means['month'] = ensemble_means['time'].dt.month


            # Calculate temporal means
            temporal_means = (ensemble_means
                               .groupby(['latitude', 'longitude', 'month', 'lead_time'])[['predicted_precip', 'precip']]
                               .mean().reset_index())

            # Calculate correlation
            temporal_means_corr = (temporal_means
                                    .groupby(['month', 'lead_time'])[['predicted_precip', 'precip']].corr(method='spearman')
                                    .iloc[0::2, -1].droplevel(-1).reset_index()) #Corrected droplevel

            # Store results in dictionaries
            temporal_means_corr_dict[df_key] = temporal_means_corr
        else:
            print(f"Warning: DataFrame not found for region '{i}' and model '{j}'")

In [16]:
# Combine all correlations into one DataFrame
temporal_corr = pd.concat(temporal_means_corr_dict.values(), keys=temporal_means_corr_dict.keys(), names=['region', 'model'])
temporal_corr = temporal_corr.rename(columns = {'precip': 'corr'}).reset_index().drop('level_2', axis=1)

In [17]:
# Makes copy of corr table
temporal_means_dm = temporal_corr.copy()

for index, row in temporal_corr.iterrows():  # Iterate over rows
    region = row['region']
    month = row['month']

    if region in dry_mask_months and month in dry_mask_months[region]:
        temporal_means_dm.loc[index, 'corr'] = np.nan  # Set 'corr' to NaN for the matching row

In [18]:
fg = sns.FacetGrid(temporal_corr, col='region', row='model', sharex=False, sharey=False)
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'corr', square = True)

fg.set_titles('Spearman Corr of Temporal Mean \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")
fg.savefig('figures/temporal_means_corr.png')
plt.close()

In [19]:
fg = sns.FacetGrid(temporal_means_dm, col='region', row='model', sharex=False, sharey=False)
fg.map_dataframe(draw_heatmap, 'lead_time', 'month', 'corr', square = True)
fg.set_titles('Spearman Corr of Temporal Mean \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Month")
fg.set_xlabels("Lead Time")
fg.savefig('figures/dry_masked_temporal_means_corr.png')
plt.close()